# AgriFM, frozen encoder (1024-d embeddings)

This notebook follows `src/models/agrifm.py` and shows how the wrapper turns the shared benchmark object into the Sentinel-2 sequence expected by AgriFM.

Key properties:
- Uses Sentinel-2 bands only.
- Resamples each parcel time series to 32 frames.
- Broadcasts each parcel-level sequence into a small spatial tile before the Swin-3D encoder.
- Pools the final encoder feature map into one embedding per parcel.

## What AgriFM expects as input

| Tensor | Shape | Meaning |
| --- | --- | --- |
| `series` | `(N, 32, 10)` | normalized Sentinel-2 band sequence |
| `pixel_values` | `(N, 32, 10, tile, tile)` | broadcast spatial tile fed to the encoder |
| output embedding | `(N, 1024)` | pooled frozen representation |

The wrapper maps benchmark Sentinel-2 band names into AgriFM order and masks missing observations before normalization.

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
from types import SimpleNamespace

import numpy as np

REPO = Path.cwd()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))


def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


In [ ]:
agrifm = load('agrifm_mod', 'src/models/agrifm.py')

print('AgriFM bands:', agrifm.AGRIFM_S2_BANDS)
print('frames:', agrifm.AGRIFM_NUM_FRAMES)
print('embedding dim:', agrifm.AGRIFM_EMBEDDING_DIM)
print('default weights:', agrifm.DEFAULT_WEIGHTS_PATH)


In [ ]:
rng = np.random.default_rng(7)
N, T = 4, 18

s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI']
s1_bands = ['VV', 'VH']
climate_bands = ['temperature', 'precipitation', 'elevation', 'slope']

bench = SimpleNamespace(
    name='synthetic',
    s2=(rng.random((N, T, len(s2_bands))) * 5000).astype('float32'),
    s1=rng.normal(-12, 3, size=(N, T, len(s1_bands))).astype('float32'),
    climate=rng.normal(0, 1, size=(N, T, len(climate_bands))).astype('float32'),
    s2_mask=np.ones((N, T), dtype='float32'),
    s1_mask=np.ones((N, T), dtype='float32'),
    climate_mask=np.ones((N, T), dtype='float32'),
    doy=np.tile(np.linspace(15, 350, T), (N, 1)).astype('float32'),
    latlon=np.repeat(np.array([[11.0, 79.0]], dtype='float32'), N, axis=0),
    years=np.full(N, 2021, dtype='int64'),
    s2_bands=s2_bands,
    s1_bands=s1_bands,
    climate_bands=climate_bands,
)

print('s2', bench.s2.shape, bench.s2_bands)
print('s1', bench.s1.shape, bench.s1_bands)
print('climate', bench.climate.shape, bench.climate_bands)
print('doy', bench.doy.shape, 'latlon', bench.latlon.shape)


## Wrapper mapping

`AgriFMEncoder.to_agrifm_series` performs the model-specific conversion without loading weights. This is the cheapest way to inspect the time resampling, band order, masking, and normalization path.

In [ ]:
enc = agrifm.AgriFMEncoder(device='cpu')
series = enc.to_agrifm_series(bench)

print('series', series.shape)
print('mean by band', np.round(series.mean(axis=(0, 1)), 3))
print('std by band', np.round(series.std(axis=(0, 1)), 3))


## Forward pass -> embedding

The real forward pass requires the downloaded AgriFM checkpoint. It is disabled by default because this model is much heavier than the conversion check above.

In [ ]:
RUN_REAL_FORWARD = False

enc = agrifm.AgriFMEncoder(device='cpu')
weights_path = enc._weights_path()

if not RUN_REAL_FORWARD:
    print('Set RUN_REAL_FORWARD = True to run the frozen encoder.')
elif not weights_path.exists():
    print('Skipping encode: weights not found at', weights_path)
else:
    emb = enc.encode(bench)
    print('embedding', emb.shape)
    print('first row, first 8 dims', np.round(emb[0, :8], 4))
